# FirstGradBooster v2

Builds on `FirstGradBooster.ipynb` by Aidan. Same data (`full_dataset.csv`), same three sensors (pH, turbidity, temperature). What's new:

1. **Grouped + temporal split** — split sites first, then drop pre-2015 rows from test sites. Honors CLAUDE.md's protocol: *"grouped + temporal split by `site_id` and date."*
2. **Always-HIGH dummy baseline** — the honest comparison point. Macro-F1 of XGBoost vs LogReg is meaningless without it given 95% class imbalance.
3. **Feature importance interpretation** — auto-report the dominant feature by gain/weight/cover, not just plot.
4. **Abstain state** — a fourth output `"send sample to lab"` when input is out of training range OR model is low-confidence. Required by CLAUDE.md deployment guardrails.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.dummy import DummyClassifier
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.metrics import classification_report, ConfusionMatrixDisplay, f1_score

from xgboost import XGBClassifier

RANDOM_STATE = 42
CUTOFF_YEAR = 2015
np.random.seed(RANDOM_STATE)

## Step 1: Load data

Same source as v1. Run from the repo root — no `os.chdir` (v1's hard-coded path is Windows-specific).

In [ ]:
df = pd.read_csv('Data/full_dataset.csv')
print(f"Shape: {df.shape}")
print(f"Sites: {df['site_id'].nunique()}")
print(f"Date range: {df['date'].min()} to {df['date'].max()}")

## Step 2: Feature prep (unchanged from v1)

Three sensors — `ph`, `turbidity_ntu`, `temperature_c`. Temperature has 45% missing → median-impute with an explicit `temperature_missing` indicator column. `ecoli_per_100ml` is excluded (the label is derived from it: target leakage).

In [ ]:
FEATURES = ['ph', 'turbidity_ntu', 'temperature_c', 'temperature_missing']
TARGET = 'risk_drinking_no_treatment'

df_model = df.copy()
df_model['temperature_missing'] = df_model['temperature_c'].isna().astype(int)
df_model['temperature_c'] = df_model['temperature_c'].fillna(df_model['temperature_c'].median())
df_model['date'] = pd.to_datetime(df_model['date'])
df_model['year'] = df_model['date'].dt.year

print(df_model[FEATURES].describe().round(2))

## Step 3: Grouped + temporal split (cutoff 2015)

v1 split sites 80/20 with `GroupShuffleSplit` — no site leak, but train and test could share *years*. v2 adds a temporal cutoff on top of the group separation:

- A site **fully** in pre-2015 → train
- A site **fully** in post-2015 → test
- A site **spanning** the cutoff → its post-2015 rows go to test; its pre-2015 rows are **dropped**

This guarantees both no-site-overlap AND temporal direction. It costs some training data — sites that span the cutoff contribute zero rows to train. The trade-off is honest: the test set genuinely represents *"future readings from sites the model has not seen."*

In [ ]:
year_counts = df_model['year'].value_counts().sort_index()
print(f"Years observed: {year_counts.index.min()} to {year_counts.index.max()}")
print(f"Rows pre-{CUTOFF_YEAR}:  {(df_model['year'] < CUTOFF_YEAR).sum():>6}")
print(f"Rows post-{CUTOFF_YEAR}: {(df_model['year'] >= CUTOFF_YEAR).sum():>6}")

fig, ax = plt.subplots(figsize=(9, 3))
colors = ['steelblue' if y < CUTOFF_YEAR else 'firebrick' for y in year_counts.index]
year_counts.plot(kind='bar', ax=ax, color=colors)
ax.set_title(f'Row count by year (blue = train candidate, red = test candidate, cutoff = {CUTOFF_YEAR})')
ax.set_xlabel('Year')
plt.tight_layout()
plt.show()

In [ ]:
# Sites with ANY rows >= cutoff are reserved for test.
test_sites = set(df_model.loc[df_model['year'] >= CUTOFF_YEAR, 'site_id'].unique())

train_df = df_model[(df_model['year'] < CUTOFF_YEAR) & (~df_model['site_id'].isin(test_sites))]
test_df  = df_model[(df_model['year'] >= CUTOFF_YEAR) & (df_model['site_id'].isin(test_sites))]

X_train, y_train = train_df[FEATURES], train_df[TARGET]
X_test,  y_test  = test_df[FEATURES],  test_df[TARGET]

print(f"Train: {len(train_df):>6} rows, {train_df['site_id'].nunique():>4} sites")
print(f"Test:  {len(test_df):>6} rows, {test_df['site_id'].nunique():>4} sites")
overlap = set(train_df['site_id']) & set(test_df['site_id'])
print(f"Site overlap (should be 0): {len(overlap)}")
print(f"\nTrain dates: {train_df['date'].min().date()} to {train_df['date'].max().date()}")
print(f"Test dates:  {test_df['date'].min().date()} to {test_df['date'].max().date()}")
print(f"\nTrain class balance (%):")
print(y_train.value_counts(normalize=True).mul(100).round(1))

## Step 4: Always-HIGH dummy baseline

The honest comparison point. With ~95% HIGH in training data, a constant-HIGH classifier scores ~95% accuracy but has zero ability to identify safe water. Any model that doesn't *beat this on macro-F1* is not contributing predictive value, regardless of its accuracy.

In [ ]:
dummy = DummyClassifier(strategy='constant', constant='HIGH')
dummy.fit(X_train, y_train)
y_pred_dummy = dummy.predict(X_test)

print("Always-HIGH baseline on test set:")
print(classification_report(y_test, y_pred_dummy, digits=3, zero_division=0))

## Step 5: XGBoost classifier (same config as v1, new split)

Same hyperparameters as v1 (300 estimators, depth 4, balanced sample weights). The only thing that changes is the training data — now using the grouped+temporal split.

In [ ]:
le = LabelEncoder()
y_train_enc = le.fit_transform(y_train)
y_test_enc  = le.transform(y_test)
print("Label encoding:", dict(zip(le.classes_, le.transform(le.classes_))))

sample_weights = compute_sample_weight('balanced', y_train_enc)

xgb_model = XGBClassifier(
    n_estimators=300,
    max_depth=4,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric='mlogloss',
    random_state=RANDOM_STATE,
    n_jobs=-1,
)
xgb_model.fit(X_train, y_train_enc, sample_weight=sample_weights)

y_pred_xgb_enc = xgb_model.predict(X_test)
y_pred_xgb = le.inverse_transform(y_pred_xgb_enc)

print("\nXGBoost on test set:")
print(classification_report(y_test, y_pred_xgb, digits=3, zero_division=0))

## Step 6: Logistic regression baseline (same config as v1, new split)

In [ ]:
lr_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', LogisticRegression(class_weight='balanced', max_iter=1000, random_state=RANDOM_STATE)),
])
lr_pipeline.fit(X_train, y_train)
y_pred_lr = lr_pipeline.predict(X_test)

print("Logistic regression on test set:")
print(classification_report(y_test, y_pred_lr, digits=3, zero_division=0))

## Step 7: Compare all three models

The number that matters: do XGBoost and LogReg *both* beat the always-HIGH dummy on macro-F1? If not, the predictive signal from these three sensors is below noise floor on this held-out distribution.

In [ ]:
models = {
    'Always-HIGH (dummy)': y_pred_dummy,
    'Logistic regression': y_pred_lr,
    'XGBoost':             y_pred_xgb,
}

rows = []
for name, y_pred in models.items():
    rep = classification_report(y_test, y_pred, output_dict=True, zero_division=0)
    rows.append({
        'model': name,
        'macro_F1':    rep['macro avg']['f1-score'],
        'recall_HIGH': rep.get('HIGH', {}).get('recall', 0.0),
        'recall_Med':  rep.get('Med',  {}).get('recall', 0.0),
        'recall_low':  rep.get('low',  {}).get('recall', 0.0),
        'accuracy':    rep['accuracy'],
    })

comparison = pd.DataFrame(rows).set_index('model').round(3)
print(comparison)

## Step 8: Feature importance with interpretation

v1 plotted gain/weight/cover but didn't say which feature won. v2 also prints the top feature by each importance type so the chart can be read in one glance.

In [ ]:
booster = xgb_model.get_booster()

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, imp_type in zip(axes, ['gain', 'weight', 'cover']):
    scores = booster.get_score(importance_type=imp_type)
    scores = {f: scores.get(f, 0) for f in FEATURES}
    sorted_scores = dict(sorted(scores.items(), key=lambda x: x[1]))
    ax.barh(list(sorted_scores.keys()), list(sorted_scores.values()))
    ax.set_title(f'Importance by {imp_type}')

plt.suptitle("XGBoost Feature Importance", fontsize=13)
plt.tight_layout()
plt.show()

print("\nDominant feature by importance type:")
for imp_type in ['gain', 'weight', 'cover']:
    scores = booster.get_score(importance_type=imp_type)
    scores = {f: scores.get(f, 0) for f in FEATURES}
    top = max(scores, key=scores.get)
    print(f"  {imp_type:>7}: {top:<22} ({scores[top]:.2f})")

## Step 9: Abstain policy (range + confidence)

CLAUDE.md deployment guardrail: *"Required output state: abstain / 'send sample to lab' when input is out of training range. No silent extrapolation."*

The implemented policy abstains when **either** condition is true:
1. Any of the three sensors reads outside the 1st–99th percentile observed in training (out-of-distribution).
2. The XGBoost model's max class probability is below 0.6 (low confidence).

This is conservative — expect ~10–20% of test rows to be flagged for lab follow-up. Trade-off: precision (more samples sent to lab) vs safety (fewer confident-but-wrong predictions in the field).

In [ ]:
# Per-feature training-distribution stats for the OOD check
train_stats = {}
for f in ['ph', 'turbidity_ntu', 'temperature_c']:
    train_stats[f] = {
        'p01': X_train[f].quantile(0.01),
        'p99': X_train[f].quantile(0.99),
    }

print("Training percentiles used for OOD detection:")
for f, s in train_stats.items():
    print(f"  {f:<18} [p01={s['p01']:.3f}, p99={s['p99']:.3f}]")


def should_abstain(row, proba, stats, conf_threshold=0.6):
    """Return True if the device should abstain instead of predicting.

    Abstain if any of pH / turbidity / temperature is outside the training
    [p01, p99] window, OR if the model's top-class probability < conf_threshold.
    """
    for f in ['ph', 'turbidity_ntu', 'temperature_c']:
        if row[f] < stats[f]['p01'] or row[f] > stats[f]['p99']:
            return True
    if proba.max() < conf_threshold:
        return True
    return False


# Apply to the held-out test set
y_pred_proba = xgb_model.predict_proba(X_test)
abstain_mask = np.array([
    should_abstain(X_test.iloc[i], y_pred_proba[i], train_stats)
    for i in range(len(X_test))
])
print(f"\nTest rows flagged for abstain: {abstain_mask.sum()} / {len(X_test)} ({abstain_mask.mean()*100:.1f}%)")

## Step 10: Final evaluation with abstain state

Build a 4-state output {HIGH, Med, low, abstain} and report:
- The 4-state confusion matrix (rows = true risk, columns = device output including abstain).
- Classification metrics on the non-abstained subset — i.e., quality of predictions when the device *commits* to an answer.
- The true-label breakdown of abstained rows — what was actually in the samples we sent to lab.

In [ ]:
y_pred_with_abstain = np.where(abstain_mask, 'abstain', y_pred_xgb)
labels_ordered = ['HIGH', 'Med', 'low', 'abstain']

fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred_with_abstain,
    labels=labels_ordered, ax=ax, colorbar=False,
)
ax.set_title("Test confusion matrix with abstain state")
plt.tight_layout()
plt.show()

print("\nMetrics on rows where the device DID NOT abstain:")
print(classification_report(
    y_test[~abstain_mask], y_pred_xgb[~abstain_mask],
    digits=3, zero_division=0,
))

print(f"\nTrue-label breakdown of the {abstain_mask.sum()} abstained rows (%):")
print(y_test[abstain_mask].value_counts(normalize=True).mul(100).round(1))

## Summary

What v2 changes vs v1:
- Grouped + temporal split (`site_id` × `year < 2015 / >= 2015`). Stricter held-out — expect lower numbers than v1's site-only split.
- Always-HIGH dummy baseline added. The macro-F1 of any honest model has to clear the dummy's macro-F1 (~0.32 expected) to be worth shipping.
- Feature importance is now reported in text, not just plotted.
- Abstain state implemented with range-based OOD detection plus low-confidence trigger. This is the CLAUDE.md *"send sample to lab"* output.

**Open questions for next iteration:**
- The OOD percentile bounds (p01/p99) are arbitrary. Tightening to p05/p95 raises abstain rate; loosening to absolute physical bounds (e.g. `0 ≤ pH ≤ 14`) lowers it. Calibrate against an operational budget.
- The confidence threshold 0.6 is a guess. The right number depends on cost asymmetry: cost of a missed-HIGH (Type II — field-deployed device says "safe" when it isn't) vs cost of a false-positive abstain (sample sent to lab unnecessarily).
- v1's regression on log E. coli scored R² = −0.17 on its grouped split. v2 inherits the same limitation: three sensors do not carry enough information to recover the E. coli load. The abstain state is the *correct design response* to this fact — see the strategic insight from the v1 review.